In [ ]:
# Import packages
from os.path import join as pjoin
import pandas as pd
import numpy as np
import osgeo
import xarray as xr
import xrspatial as xrs
import rioxarray
import rasterstats as rs
import os
os.environ['USE_PYGEOS'] = '0'
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import fiona

# directory settings
input_dir = '../input'   # relative path to the input data
scratch_dir = '../scratch'   # relative path to the output data.

print(os.getcwd())

The first three cells of code contain code to load the global PCR-GLOBWB 2.0 output files, and to slice them across time and space, namely obtaining monthly values from January 2010 untill December 2019 and into a coordinate window covering only South Africa. This part of the code is now commented out, since the data provided is already the output from this process. 

In [ ]:
# # global PCR-GLOB output files
# global_discharge_file = 'pcrglobwb_cmip6-isimip3-gswp3-w5e5_image-aqueduct_historical-reference_discharge_global_monthly-average_1960_2019_basetier1.nc'
# global_runoff_file = 'pcrglobwb_cmip6-isimip3-gswp3-w5e5_image-aqueduct_historical-reference_runoff_global_monthly-total_1960_2019_basetier1.nc'
# global_gw_recharge_file = 'pcrglobwb_cmip6-isimip3-gswp3-w5e5_image-aqueduct_historical-reference_gwRecharge_global_monthly-total_1960_2019_basetier1.nc'
# global_desalination_file = 'pcrglobwb_cmip6-isimip3-gswp3-w5e5_image-aqueduct_historical-reference_desalinationAbstraction_global_monthly-total_1960_2019_basetier1.nc'

# Qsw_global = rioxarray.open_rasterio(pjoin(input_dir, global_discharge_file))
# Qro_global = rioxarray.open_rasterio(pjoin(input_dir, global_runoff_file))
# Qgwr_global = rioxarray.open_rasterio(pjoin(input_dir, global_gw_recharge_file))
# Qds_global = rioxarray.open_rasterio(pjoin(input_dir, global_desalination_file))

# Qsw_global.dims

In [ ]:
# select SA
import cftime

xmin, xmax = 12, 35.5
ymin, ymax = -35, -22
tmin = cftime.DatetimeGregorian(2010, 1, 1, 0, 0, 0, 0, has_year_zero=False)
tmax = cftime.DatetimeGregorian(2019, 12, 31, 0, 0, 0, 0, has_year_zero=False)

# Qsw = Qsw_global.sel(x=slice(xmin, xmax), y=slice(ymax, ymin), time=slice(tmin, tmax))
# Qro = Qro_global.sel(x=slice(xmin, xmax), y=slice(ymax, ymin), time=slice(tmin, tmax))
# Qgwr = Qgwr_global.sel(x=slice(xmin, xmax), y=slice(ymax, ymin), time=slice(tmin, tmax))
# Qds = Qds_global.sel(x=slice(xmin, xmax), y=slice(ymax, ymin), time=slice(tmin, tmax))

# Qsw

In [ ]:
# # SA file names of PCR GLOB output variables
# sa_discharge_file = 'pcrglobwb_cmip6-isimip3-gswp3-w5e5_image-aqueduct_historical-reference_discharge_SA_monthly-average_2010_2019_basetier1.nc'
# sa_runoff_file = 'pcrglobwb_cmip6-isimip3-gswp3-w5e5_image-aqueduct_historical-reference_runoff_SA_monthly-total_2010_2019_basetier1.nc'
# sa_gw_recharge_file = 'pcrglobwb_cmip6-isimip3-gswp3-w5e5_image-aqueduct_historical-reference_gwRecharge_SA_monthly-total_2010_2019_basetier1.nc'
# sa_desalination_file = 'pcrglobwb_cmip6-isimip3-gswp3-w5e5_image-aqueduct_historical-reference_desalinationAbstraction_SA_monthly-total_2010_2019_basetier1.nc'

# # save sliced data to separate dataset
# Qsw.to_netcdf(pjoin(scratch_dir, sa_discharge_file))
# Qro.to_netcdf(pjoin(scratch_dir, sa_runoff_file))
# Qgwr.to_netcdf(pjoin(scratch_dir, sa_gw_recharge_file))
# Qds.to_netcdf(pjoin(scratch_dir, sa_desalination_file))

In [ ]:
# SA file names of PCR GLOB output variables
sa_discharge_file = 'pcrglobwb_cmip6-isimip3-gswp3-w5e5_image-aqueduct_historical-reference_discharge_SA_monthly-average_2010_2019_basetier1.nc'
sa_runoff_file = 'pcrglobwb_cmip6-isimip3-gswp3-w5e5_image-aqueduct_historical-reference_runoff_SA_monthly-total_2010_2019_basetier1.nc'
sa_gw_recharge_file = 'pcrglobwb_cmip6-isimip3-gswp3-w5e5_image-aqueduct_historical-reference_gwRecharge_SA_monthly-total_2010_2019_basetier1.nc'
sa_desalination_file = 'pcrglobwb_cmip6-isimip3-gswp3-w5e5_image-aqueduct_historical-reference_desalinationAbstraction_SA_monthly-total_2010_2019_basetier1.nc'

Qsw = xr.open_dataset(pjoin(scratch_dir, sa_discharge_file))
Qro = xr.open_dataset(pjoin(scratch_dir, sa_runoff_file))
Qgw = xr.open_dataset(pjoin(scratch_dir, sa_gw_recharge_file))
Qds = xr.open_dataset(pjoin(scratch_dir, sa_desalination_file))

# the xarray stores the "spatial_ref" as separate dimension, so open only the usefull variable)
Qsw = Qsw[list(Qsw.data_vars)[1]]
Qro = Qro[list(Qro.data_vars)[1]]
Qgw = Qgw[list(Qgw.data_vars)[1]]
Qds = Qds[list(Qds.data_vars)[1]]

# Re-enable rioxarray access
Qsw = Qsw.rio.write_crs("EPSG:4326") 
Qro = Qro.rio.write_crs("EPSG:4326") 
Qgw = Qgw.rio.write_crs("EPSG:4326") 
Qds = Qds.rio.write_crs("EPSG:4326") 

In [ ]:
# compute the multi-year mean of our slice (2010-2019)
Qsw_mean = Qsw.mean(dim='time')
Qro_mean = Qro.mean(dim='time')
Qgw_mean = Qgw.mean(dim='time')
Qds_mean = Qds.mean(dim='time')

# Qsw.plot.imshow(col='time', col_wrap=3, vmax=50, add_colorbar=False)
# Qannual.plot.imshow(col='time', col_wrap=3, vmax=50, add_colorbar=False)
# Qsw.plot.imshow(vmax=50, cmap='viridis', add_colorbar=True)

# Qsw_mean, Qro_mean, Qgw_mean, Qds_mean
# Qsw_mean.plot.imshow(vmax=None, cmap='viridis', add_colorbar=True)

In [ ]:
# import pcraster
import rasterio 

file = pjoin(input_dir, "cellAreaGlob.map")

# using rioxarray to open .map file
cell_area = rioxarray.open_rasterio(file).squeeze("band", drop=True)
cell_area_SA = cell_area.sel(x=slice(xmin, xmax), y=slice(ymax, ymin))

# using pcraster to open .map file
# cell_area = pcraster.readmap(file)
cell_area_SA.plot.imshow(vmax=None, cmap='viridis', add_colorbar=True)
cell_area_SA.rio.write_crs(4326, inplace=True)
cell_area_SA.shape

In [ ]:
# multiply the data in [m / month] with the area of each cell (raster, pixel by pixel multiplication) in [m2] to obtain [m3 / month]
Qro_mean_m3 = Qro_mean.data * cell_area_SA.data
Qgw_mean_m3 = Qgw_mean.data * cell_area_SA.data
Qds_mean_m3 = Qds_mean.data * cell_area_SA.data

Qro_mean_m3 = xr.DataArray(data=Qro_mean_m3, dims=Qro_mean.dims, coords=Qro_mean.coords, attrs=Qro_mean.attrs, name="land_surface_runoff_mean_monthly_volume")
Qgw_mean_m3 = xr.DataArray(data=Qgw_mean_m3, dims=Qgw_mean.dims, coords=Qgw_mean.coords, attrs=Qgw_mean.attrs, name="groundwater_recharge_mean_monthly_volume")
Qds_mean_m3 = xr.DataArray(data=Qds_mean_m3, dims=Qds_mean.dims, coords=Qds_mean.coords, attrs=Qds_mean.attrs, name="desalination_source_abstraction_mean_monthly_volume")

# multiply the data in [m3 / s] with the average number of seconds per month to obtain [m3 / month]
Qsw_mean_m3 = Qsw_mean * 60 * 60 * 24 * 30.4

Qro_mean_m3

In [ ]:
plt.imshow(Qro_mean_m3)

In [ ]:
plt.imshow(Qgw_mean_m3)

In [ ]:
plt.imshow(Qds_mean_m3)

In [ ]:
Qro_mean_m3

In [ ]:
# get vector data for zones from e.g. municipalities
# 2011 municipal boundaries
mun11 = gpd.read_file(pjoin(input_dir, 'MN_2011.shp'))
print(mun11.crs) # note identical crs as the raster data

In [ ]:
#2016 municipal boundaries
gdb = pjoin(input_dir, 'MN_2016.gdb')

fiona.listlayers(gdb)

mun16 = gpd.read_file(filename = gdb, layer='MDBLocalMunicipalBoundary2016')
print(mun16.crs)

In [ ]:
# The zonal_stats is focused on files, but here an implementation in memory is done:
variables_sum = ['Qro_mean_m3', 'Qgw_mean_m3', 'Qds_mean_m3']
variables_max = ['Qsw_mean_m3']

for mun in [mun11, mun16]:
    for variable_name in variables_sum:
        variable = eval(variable_name)  # this keeps it a DataArray
        raster = variable.isel().data
        aff = variable.rio.transform() # The affine transformatiion scales, translates, numpy values to positions on the earth.
        stats = rs.zonal_stats(mun.geometry, raster, affine=aff, stats=["sum"]) # generates a list of dictionaries (not so intuitive)
        mun[f'{variable_name}_total'] = [s["sum"] for s in stats]

for mun in [mun11, mun16]:
    for variable_name in variables_max:
        variable = eval(variable_name)  # this keeps it a DataArray
        raster = variable.isel().data
        aff = variable.rio.transform() # The affine transformatiion scales, translates, numpy values to positions on the earth.
        stats = rs.zonal_stats(mun.geometry, raster, affine=aff, stats=["max"]) # generates a list of dictionaries (not so intuitive)
        mun[f'{variable_name}_max'] = [s["max"] for s in stats]      
        
mun16.explore(column='Qsw_mean_m3_max')

In [ ]:
Qsw_mean_m3.isel().plot.imshow(add_colorbar=True)

In [ ]:
# save to disc for inspection in QGIS
# Qmon.isel(time=1).rio.to_raster(pjoin(scratch_dir, 'Qmon_JAN.tif'))
# mun11.to_file(pjoin(scratch_dir, 'municipal_water_availability_2010_2019_demarcations11.gpkg'))
mun16.to_file(pjoin(scratch_dir, 'municipal_water_availability_2010_2019.gpkg'))

In [ ]:
# check spatial alignment
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 8))
Qsw_mean_m3.plot(ax=ax, cmap="viridis")
mun16.boundary.plot(ax=ax, color="red", linewidth=0.5)
plt.show()